# Data Snapshot Metadata Extraction

Run `build_openai_schema` only when the canonical metadata field map changes. This notebook reuses the generated OpenAI Structured Outputs schema for every snapshot.

In [1]:
%load_ext autotime

import json
import os
import re
import time
from pathlib import Path

from tqdm.auto import tqdm

from data_snapshot.metadata_extraction import extract_metadata
from data_snapshot.constants import ROOT

## Inputs

In [2]:
SOURCE = "unhcr"
SLEEP_SECONDS = 0.2
SNAPSHOTS_DIR = Path("data/snapshots")
DOCUMENT_METADATA_DIR = Path("data/document_metadata")
OPENAI_SCHEMA_PATH = ROOT / "src/data_snapshot/metadata_extraction/schema/openai_schema_v1.1.json"
CONFIG_PATH = ROOT / "src/data_snapshot/metadata_extraction/config/default.json"
OUTPUT_JSONL_PATH = Path(f"outputs/extracted_metadata_{SOURCE}.jsonl")
LOG_JSONL_PATH = Path(f"outputs/extraction_calls_{SOURCE}.jsonl")

In [3]:
snapshot_files = sorted(SNAPSHOTS_DIR.glob("*.png"))
metadata_lookup = {
    path.name.removesuffix("_metadata.json"): path
    for path in DOCUMENT_METADATA_DIR.glob("*_metadata.json")
}

print(f"Snapshots found: {len(snapshot_files)}")
print(f"Metadata files found: {len(metadata_lookup)}")

Snapshots found: 4
Metadata files found: 2


## Main pipeline

In [4]:
# # Smoke test
# snapshot_files = snapshot_files[0:1]

# len(snapshot_files)

In [5]:
# Create skip list
completed_images = set()
if OUTPUT_JSONL_PATH.exists():
    with OUTPUT_JSONL_PATH.open(encoding="utf-8") as file:
        completed_images = {json.loads(line)["image_name"] for line in file}

for snapshot_path in tqdm(snapshot_files):
    if snapshot_path.name in completed_images:
        print(f"Skipped: {snapshot_path.name}")
        continue

    document_id = re.sub(r"_(figure|table)_\d+$", "", snapshot_path.stem)
    result = extract_metadata(
        image_path=snapshot_path,
        openai_schema_path=OPENAI_SCHEMA_PATH,
        output_jsonl_path=OUTPUT_JSONL_PATH,
        log_jsonl_path=LOG_JSONL_PATH,
        config_path=CONFIG_PATH,
        source_document_metadata_path=metadata_lookup.get(document_id),
        source=SOURCE,
    )
    if result.error:
        print(f"{snapshot_path.name}: {result.error}")
    time.sleep(SLEEP_SECONDS)

  0%|          | 0/4 [00:00<?, ?it/s]

Skipped: 1_advocacy_note_mineaction_-_niger_eng_figure_000.png


In [6]:
if LOG_JSONL_PATH.exists():
    with LOG_JSONL_PATH.open(encoding="utf-8") as file:
        log_rows = [json.loads(line) for line in file]
    print(f"API calls: {len(log_rows)}")
    print(f"Errors: {sum(row['error'] is not None for row in log_rows)}")

API calls: 4
Errors: 0
